In [ ]:
from common.setup_plotting import setup_matplotlib, get_figure_dir
from common.data_downloader import download_data
from common.data import get_dataloaders, preprocess_spectra, normalize, denormalize, SpectraDataset
from common.models import SpectraCNNFlowEncoder
from B03train_normalizing_flow_template import CombinedModel
from common.losses import nf_loss
from helper import run_prediction_diagnostics
from common.training import train_model, compute_flow_loss

import random
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score

setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks 
%matplotlib inline  
import matplotlib.pyplot as plt   # THEN import pyplot

In [ ]:
# download data for this task, if needed and get path 
data_path = download_data("task_01") # Task 01 data is used for task 02 as well, so we can reuse it here
# get and if needed create figure directory for this task
fig_dir = get_figure_dir("task_03")

In [ ]:
spectra = np.load(f"{data_path}/spectra.npy")
spectra_length = spectra.shape[1]

# labels: mass, age, l_bol, dist, t_eff, log_g, fe_h, SNR
labelNames = ["mass", "age", "l_bol", "dist", "t_eff", "log_g", "fe_h", "SNR"]
labels = np.load(f"{data_path}/labels.npy")

# We only use the three labels: t_eff, log_g, fe_h
labelNames = labelNames[-4:-1]
labels = labels[:, -4:-1]
n_labels = labels.shape[1]


# preprocess spectra
spectra = preprocess_spectra(spectra)

# normalize spectra
spectra_norm, spectra_mean, spectra_std = normalize(spectra)

# normalize labels column-wise
labels_norm, label_mean, label_std = normalize(
    labels,
    axis=0,
)

# create dataset
dataset = SpectraDataset(
    spectra_norm,
    labels_norm,
)

# create dataloaders
train_loader, val_loader, test_loader = get_dataloaders(
    dataset,
    batch_size=64,
)

batch_spectra, batch_labels = next(iter(train_loader))

print("batch_spectra shape:", batch_spectra.shape)
print("batch_labels shape:", batch_labels.shape)
print("batch_spectra dtype:", batch_spectra.dtype)
print("batch_labels dtype:", batch_labels.dtype)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [ ]:
skip_training = True  # Set to True to skip training and load existing checkpoints, False to train models from scratch

def flow_nll_loss(model, batch_x, batch_y):
    return nf_loss(
        inputs=batch_x,
        batch_labels=batch_y,
        model=model,
    )


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

flow_model_configs = {
    "diagonal_gaussian": {
        "nf_type": "diagonal_gaussian",
        "checkpoint_path": "../models/diagonal_gaussian_flow.pt",
    },
    "full_gaussian": {
        "nf_type": "full_gaussian",
        "checkpoint_path": "../models/full_gaussian_flow.pt",
    },
    "full_flow": {
        "nf_type": "full_flow",
        "checkpoint_path": "../models/full_flow.pt",
    },
}

trained_models = {}
training_histories = {}

base_seed = 42

for model_idx, (model_name, config) in enumerate(flow_model_configs.items()):
    set_seed(base_seed + model_idx)

    print(f"\nTraining {model_name} model")
    print("-" * 80)

    encoder = lambda latent_dimension: SpectraCNNFlowEncoder(
        latent_dimension=latent_dimension,
        channels=(16, 32, 64),
        kernel_size=3,
        convs_per_block=2,
        pool_size=4,
        adaptive_pool_length=64,
        hidden_dims=(128,),
        dropout=0.0,
        use_batchnorm=True,
    )

    model = CombinedModel(
        encoder=encoder,
        nf_type=config["nf_type"],
    )

    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
    )

    train_losses, val_losses = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=flow_nll_loss,
        device=device,
        num_epochs=100,
        checkpoint_path=config["checkpoint_path"],
        patience=10,
        compute_loss_fn=compute_flow_loss,
        skip_training=skip_training
    )

    trained_models[model_name] = model

    training_histories[model_name] = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "checkpoint_path": config["checkpoint_path"],
    }

In [ ]:
latex_names = [
    r"$T_{\mathrm{eff}}$",
    r"$\log g$",
    r"$[\mathrm{Fe}/\mathrm{H}]$",
]

all_diagnostic_results = {}

for model_name, current_model in trained_models.items():
    print(f"\nRunning diagnostics for {model_name}")
    print("-" * 80)

    model_fig_dir = Path(fig_dir) / model_name
    model_fig_dir.mkdir(parents=True, exist_ok=True)

    results = run_prediction_diagnostics(
        model=current_model,
        test_loader=test_loader,
        device=device,
        fig_dir=model_fig_dir,
        label_mean=label_mean,
        label_std=label_std,
        n_labels=n_labels,
        latex_names=latex_names,
        samplesize_model=1000,
        samplesize_pdf=5000,
        pdf_star_indices=range(5),
        make_pdf_predictions=True,
        make_predictions_vs_true=True,
        make_uncertainty_vs_error=True,
        make_residual_distributions=True,
        show=True,
    )

    all_diagnostic_results[model_name] = results